# Specific Test IV — Neural Operator Classifier (FNO Hybrid)

Hybrid **EfficientNet-B0 + Fourier Neural Operator** classifier for gravitational lensing.
The FNO branch operates in function space via spectral convolutions (FFT), augmenting the pretrained CNN feature extractor as required by the task.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/dataset'):
    os.system('unzip -q /content/drive/MyDrive/dataset.zip -d /content/')
    print('Dataset ready')
else:
    print('Dataset already extracted')

In [ ]:
import os, json, time, shutil, glob
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import tqdm

TRAIN_DIR     = '/content/dataset/train'
VAL_DIR       = '/content/dataset/val'
SAVE_DIR      = 'fno_checkpoints'
DRIVE_OUT     = '/content/drive/MyDrive/fno_results'
CLASS_NAMES   = ['no', 'sphere', 'vort']

IMG_SIZE      = 64
FNO_WIDTH     = 48
FNO_MODES     = 12
FNO_BLOCKS    = 4
EPOCHS        = 70
WARMUP        = 5
BATCH_SIZE    = 128
LR            = 2e-4
WEIGHT_DECAY  = 5e-4
WORKERS       = 2
SAVE_EVERY    = 5

os.makedirs(SAVE_DIR,  exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
for cls in CLASS_NAMES:
    p = glob.glob(f'{TRAIN_DIR}/{cls}/*.npy')[0]
    a = np.load(p)
    print(f'{cls}: shape={a.shape} min={a.min():.3f} max={a.max():.3f}')

In [ ]:
class LensDataset(Dataset):
    def __init__(self, root, class_names, img_size=64, augment=False):
        self.img_size = img_size
        self.augment  = augment
        self.samples  = []
        for idx, cls in enumerate(class_names):
            for f in sorted(glob.glob(os.path.join(root, cls, '*.npy'))):
                self.samples.append((f, idx))

    def __len__(self):
        return len(self.samples)

    def _load(self, path):
        arr = np.load(path).astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        if arr.ndim == 2:
            arr = arr[np.newaxis]
        elif arr.ndim == 3 and arr.shape[0] != 1:
            if arr.shape[0] == 3:
                arr = (0.299*arr[0] + 0.587*arr[1] + 0.114*arr[2])[np.newaxis]
            elif arr.shape[2] == 3:
                arr = (0.299*arr[:,:,0] + 0.587*arr[:,:,1] + 0.114*arr[:,:,2])[np.newaxis]
            elif arr.shape[2] == 1:
                arr = arr.transpose(2,0,1)
        t = torch.from_numpy(arr)
        if t.shape[1] != self.img_size or t.shape[2] != self.img_size:
            t = F.interpolate(t.unsqueeze(0), size=(self.img_size, self.img_size),
                              mode='bilinear', align_corners=False).squeeze(0)
        return t  # (1, H, W) in [0, 1]

    def _augment(self, t):
        if torch.rand(1) > 0.5: t = torch.flip(t, [2])
        if torch.rand(1) > 0.5: t = torch.flip(t, [1])
        t = torch.rot90(t, k=torch.randint(0,4,(1,)).item(), dims=[1,2])
        t = torch.clamp(t + 0.02 * torch.randn_like(t), 0.0, 1.0)
        return t

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        t = self._load(path)
        if self.augment:
            t = self._augment(t)
        return t, label

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        scale = 1.0 / (in_ch * out_ch)
        self.W1    = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, modes, dtype=torch.cfloat))
        self.W2    = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, modes, dtype=torch.cfloat))
        self.modes = modes

    def forward(self, x):
        B, C, H, W = x.shape
        m  = min(self.modes, H // 2)
        ft = torch.fft.rfft2(x, norm='ortho')
        out = torch.zeros(B, self.W1.shape[1], H, W//2+1, dtype=torch.cfloat, device=x.device)
        out[:,:,:m,:m]  = torch.einsum('bixy,ioxy->boxy', ft[:,:,:m,:m],  self.W1[:,:,:m,:m])
        out[:,:,-m:,:m] = torch.einsum('bixy,ioxy->boxy', ft[:,:,-m:,:m], self.W2[:,:,:m,:m])
        return torch.fft.irfft2(out, s=(H, W), norm='ortho')


class FNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes)
        self.bypass   = nn.Conv2d(width, width, 1)
        self.norm     = nn.GroupNorm(min(8, width), width)

    def forward(self, x):
        return x + F.gelu(self.norm(self.spectral(x) + self.bypass(x)))


IMGNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
IMGNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)


class HybridFNOClassifier(nn.Module):
    def __init__(self, width=48, modes=12, n_blocks=4, n_classes=3):
        super().__init__()
        effnet = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.cnn_branch = nn.Sequential(effnet.features, nn.AdaptiveAvgPool2d(1), nn.Flatten())

        self.lift     = nn.Conv2d(1, width, 1)
        self.blocks   = nn.Sequential(*[FNOBlock(width, modes) for _ in range(n_blocks)])
        self.fno_pool = nn.Sequential(nn.AdaptiveAvgPool2d(4), nn.Flatten())
        self.head     = nn.Sequential(
            nn.Linear(1280 + width*16, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),             nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

    def train(self, mode=True):
        super().train(mode)
        # keep BN stats frozen when cnn_branch is not being trained
        if not any(p.requires_grad for p in self.cnn_branch.parameters()):
            self.cnn_branch.eval()
        return self

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, x):
        x3  = x.repeat(1, 3, 1, 1)
        mn  = IMGNET_MEAN.to(x.device)
        st  = IMGNET_STD.to(x.device)
        cnn = self.cnn_branch((x3 - mn) / st)
        fno = self.fno_pool(self.blocks(self.lift((x - 0.5) / 0.5)))
        return self.head(torch.cat([cnn, fno], dim=1))

In [ ]:
train_ds = LensDataset(TRAIN_DIR, CLASS_NAMES, IMG_SIZE, augment=True)
val_ds   = LensDataset(VAL_DIR,   CLASS_NAMES, IMG_SIZE, augment=False)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True)

steps_per_epoch = len(train_loader)
print(f'train={len(train_ds):,}  val={len(val_ds):,}  steps/epoch={steps_per_epoch}')

model = HybridFNOClassifier(FNO_WIDTH, FNO_MODES, FNO_BLOCKS).to(device)

for p in model.cnn_branch.parameters():
    p.requires_grad = False
print(f'EfficientNet frozen for {WARMUP} epochs')
print(f'Trainable params: {model.count_params():,}')

optimizer = torch.optim.AdamW([
    {'params': model.cnn_branch.parameters(), 'lr': LR * 0.1},
    {'params': list(model.lift.parameters()) + list(model.blocks.parameters()) +
               list(model.fno_pool.parameters()) + list(model.head.parameters()), 'lr': LR},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR*0.1, LR],
    total_steps=EPOCHS * steps_per_epoch,
    pct_start=0.15, anneal_strategy='cos', div_factor=25.0, final_div_factor=1e4,
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
print('Ready.')

In [ ]:
def train_epoch(epoch):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Ep {epoch:03d}/{EPOCHS} [train]', leave=False,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}')
    t0 = time.time()
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()
        bs        = labels.size(0)
        loss_sum += loss.item() * bs
        correct  += (logits.detach().argmax(1) == labels).sum().item()
        total    += bs
        pbar.set_postfix(loss=f'{loss_sum/total:.4f}', acc=f'{correct/total:.3f}',
                         lr=f'{scheduler.get_last_lr()[-1]:.1e}', t=f'{time.time()-t0:.0f}s')
    return loss_sum / total, correct / total


@torch.no_grad()
def val_epoch():
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    probs_all, labels_all = [], []
    pbar = tqdm(val_loader, desc='           [ val]', leave=False,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}')
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        probs  = torch.softmax(logits, 1)
        bs        = labels.size(0)
        loss_sum += loss.item() * bs
        correct  += (probs.argmax(1) == labels).sum().item()
        total    += bs
        probs_all.append(probs.cpu().numpy())
        labels_all.append(labels.cpu().numpy())
        pbar.set_postfix(loss=f'{loss_sum/total:.4f}', acc=f'{correct/total:.3f}')
    p = np.concatenate(probs_all)
    l = np.concatenate(labels_all)
    auc_score = roc_auc_score(label_binarize(l, classes=[0,1,2]), p, average='macro', multi_class='ovr')
    return loss_sum / total, correct / total, auc_score, p, l

In [ ]:
best_auc, best_epoch = 0.0, 0
best_probs = best_labels = None
ckpt_path    = f'{SAVE_DIR}/best_model.pt'
history_path = f'{SAVE_DIR}/history.json'
preds_path   = f'{SAVE_DIR}/val_preds.npz'
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'val_auc':[]}

print(f'{"Ep":>4} {"T-Loss":>8} {"T-Acc":>7} {"V-Loss":>8} {"V-Acc":>7} {"V-AUC":>8} {"Time":>7}  Note')
print('-' * 72)
run_start = time.time()

for epoch in range(1, EPOCHS + 1):
    if epoch == WARMUP + 1:
        for p in model.cnn_branch.parameters():
            p.requires_grad = True
        print(f'\n[phase] EfficientNet unfrozen at epoch {epoch}')

    t0 = time.time()
    tl, ta          = train_epoch(epoch)
    vl, va, vauc, probs, labels = val_epoch()

    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    history['val_auc'].append(vauc)

    elapsed = time.time() - run_start
    eta     = elapsed / epoch * (EPOCHS - epoch)
    note    = ''

    if vauc > best_auc:
        best_auc, best_epoch = vauc, epoch
        best_probs, best_labels = probs.copy(), labels.copy()
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_auc': best_auc}, ckpt_path)
        shutil.copy(ckpt_path, f'{DRIVE_OUT}/best_model.pt')
        note = '<< best'

    print(f'{epoch:4d} {tl:8.4f} {ta:7.4f} {vl:8.4f} {va:7.4f} {vauc:8.4f} {time.time()-t0:6.1f}s  {note}  ETA:{eta/60:.0f}m')

    if epoch % SAVE_EVERY == 0:
        with open(history_path, 'w') as f: json.dump(history, f)
        np.savez(preds_path, probs=best_probs, labels=best_labels)
        shutil.copy(history_path, f'{DRIVE_OUT}/history.json')
        shutil.copy(preds_path,   f'{DRIVE_OUT}/val_preds.npz')
        print(f'  [drive] epoch={epoch}  best_auc={best_auc:.4f}')

with open(history_path, 'w') as f: json.dump(history, f)
np.savez(preds_path, probs=best_probs, labels=best_labels)
for fn in ['best_model.pt', 'history.json', 'val_preds.npz']:
    shutil.copy(f'{SAVE_DIR}/{fn}', f'{DRIVE_OUT}/{fn}')

print(f'\nDone in {(time.time()-run_start)/60:.1f} min  |  Best AUC={best_auc:.4f} @ epoch {best_epoch}')

In [ ]:
n_classes  = len(CLASS_NAMES)
labels_bin = label_binarize(best_labels, classes=list(range(n_classes)))
colors     = ['#e41a1c', '#377eb8', '#4daf4a']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
per_auc = []
for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], best_probs[:, i])
    a = auc(fpr, tpr)
    per_auc.append(a)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={a:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set(xlabel='FPR', ylabel='TPR', title=f'ROC Curves (Macro AUC={np.mean(per_auc):.4f})')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

ax = axes[1]
cm   = confusion_matrix(best_labels, best_probs.argmax(1))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fno_roc_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nPer-class AUC:')
for cls, a in zip(CLASS_NAMES, per_auc): print(f'  {cls}: {a:.4f}')
print(f'  Macro: {np.mean(per_auc):.4f}')

In [ ]:
epochs_range = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val',   color='tomato')
axes[0].set(title='Loss', xlabel='Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], label='Train', color='steelblue')
axes[1].plot(epochs_range, history['val_acc'],   label='Val',   color='tomato')
axes[1].set(title='Accuracy', xlabel='Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['val_auc'], label='FNO Hybrid', color='darkorange', lw=2)
axes[2].axhline(y=0.9741, color='gray', linestyle='--', lw=1.5, label='EfficientNet-B0 baseline')
axes[2].set(title='Validation AUC', xlabel='Epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fno_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparison and Discussion

### Architecture

Two parallel branches process each image independently:

- **FNO branch**: 4 FNO blocks (width=48, modes=12). Each block applies learned linear maps in the truncated Fourier basis via  → complex weights → . Global receptive field from the first layer.
- **CNN branch**: EfficientNet-B0 (ImageNet pretrained, frozen during warmup). 1-channel input replicated to 3 channels; ImageNet normalization applied inside . Outputs 1280-dim features.

Features are concatenated and passed through a 3-layer MLP head.

### FNO vs Standard CNN

| | FNO branch | EfficientNet-B0 |
|---|---|---|
| Receptive field | Global (layer 1) | Grows with depth |
| Frequency handling | Explicit per-mode weights | Implicit via stacking |
| Pretraining | Trained from scratch | ImageNet pretrained |
| Inductive bias | Periodic translation equivariance | Local translation equivariance |

### Why Hybrid

A pure FNO trained from scratch lacks inductive bias for local features. EfficientNet provides strong hierarchical features immediately, while the FNO branch targets global spectral structure — Einstein ring morphology and substructure-induced frequency perturbations.

Different substructure types have distinct spectral signatures: smooth rings concentrate power in low modes, subhalo perturbations add mid-frequency distortions, and vortex substructure produces specific angular-frequency patterns.

### Results

| Task | Architecture | AUC |
|---|---|---|
| Common Test I | EfficientNet-B0 standalone | **0.9741** |
| Specific Test IV | EfficientNet-B0 + FNO hybrid | **see above** |

The FNO branch trains from scratch (no pretrained FNO exists for scientific imaging), which accounts for the gap relative to the pure CNN baseline.
